# Notebook 03 — Tiny R1-Zero-Style GRPO Run

**Chapter**: [Chapter 5 — RL for Reasoning](../chapters/05-rl-for-reasoning.md).

**Claim demonstrated**: GRPO with a verifiable reward (exact-match on math answer) increases chain length and accuracy on GSM8K-style problems even on a tiny model with no SFT seed — a scaled-down qualitative R1-Zero.

**Important caveat**: a faithful R1-Zero reproduction needs hundreds of GPU-hours. This notebook demonstrates the *signal*: in ~ 1 hour on a single 16-GB GPU, you can see reward go up, chain length grow, and the format reward become gameable. Treat the numbers as illustrative.

**Hardware**: ≥ 16 GB GPU. CPU is impractical.

**Dependencies**: `trl >= 0.12`, `transformers`, `peft`, `datasets`, `bitsandbytes`, `wandb` (optional).

**Fuller hosted run**: TODO — link a Modal / HF Spaces run with a 7B base.

---

## Recipe

1. Base: Qwen2.5-0.5B-Instruct (tiny enough to fit on 16 GB with bf16).
2. Task: GSM8K math problems with extractable integer answers.
3. Reward: `1.0` if extracted answer matches gold, else `0.0`. (Optional small format-bonus for emitting `\boxed{}`.)
4. Algorithm: GRPO via `trl.GRPOTrainer`.
5. Logging: per-step reward, mean chain length, accuracy on a held-out 100-problem set every 50 steps.

Expected qualitative outcome over ~ 500 steps:
- Reward starts near base-model GSM8K accuracy.
- Chain length grows from ~ 80 tokens to several hundred.
- Accuracy increases monotonically with high variance.
- Some completions exhibit format reward hacking (emitting `\boxed{}` repeatedly with random integers).

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
OUT_DIR = "./tiny-r1zero"
MAX_STEPS = 500
BATCH = 4
GROUP = 4  # GRPO group size
MAX_PROMPT_LEN = 256
MAX_COMPL_LEN = 512
LR = 1e-6
SEED = 42

import os, re, random, json
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# Data: GSM8K, formatted as a chat prompt that asks for \boxed{} final answer.
raw = load_dataset("openai/gsm8k", "main", split="train")
GOLD_RE = re.compile(r"####\s*(-?\d+)")

def format_example(ex):
    m = GOLD_RE.search(ex["answer"])
    gold = int(m.group(1)) if m else None
    prompt = (
        f"Solve the problem. Put your final integer answer in \\boxed{{}}.\n\n"
        f"{ex['question']}\n"
    )
    return {"prompt": prompt, "gold": gold}

ds = raw.map(format_example).filter(lambda x: x["gold"] is not None)
print(f"Train size: {len(ds)}")

In [ ]:
# Reward function: 1 if \boxed{N} matches gold, 0 otherwise.
BOXED_RE = re.compile(r"\\boxed\{(-?\d+)\}")

def reward_fn(prompts, completions, **kwargs):
    """trl GRPO reward signature: lists of strings, returns list[float]."""
    golds = kwargs.get("gold", [None] * len(prompts))
    rewards = []
    for c, g in zip(completions, golds):
        m = BOXED_RE.findall(c)
        try:
            pred = int(m[-1]) if m else None
            r = 1.0 if pred == g else 0.0
        except (ValueError, IndexError):
            r = 0.0
        # Small format bonus: presence of \boxed at all.
        if m:
            r += 0.05
        rewards.append(r)
    return rewards

In [ ]:
# Trainer.
cfg = GRPOConfig(
    output_dir=OUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    num_generations=GROUP,
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_COMPL_LEN,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=200,
    bf16=True,
    seed=SEED,
    report_to="none",
)
trainer = GRPOTrainer(
    model=BASE,
    reward_funcs=reward_fn,
    args=cfg,
    train_dataset=ds,
)
trainer.train()

In [ ]:
# Quick eval on a held-out slice.
eval_ds = load_dataset("openai/gsm8k", "main", split="test").select(range(100))
tokenizer = AutoTokenizer.from_pretrained(BASE)
policy = trainer.model
policy.eval()

correct = 0
for ex in eval_ds:
    gold_m = GOLD_RE.search(ex["answer"])
    if not gold_m: continue
    gold = int(gold_m.group(1))
    prompt = f"Solve the problem. Put your final integer answer in \\boxed{{}}.\n\n{ex['question']}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(policy.device)
    with torch.inference_mode():
        out = policy.generate(**inputs, max_new_tokens=MAX_COMPL_LEN, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = BOXED_RE.findall(gen)
    pred = int(m[-1]) if m and m[-1].lstrip('-').isdigit() else None
    correct += int(pred == gold)
print(f"Eval accuracy: {correct}/100")

## Notes on what you'll see (and won't)

- **Will see**: reward curve rises, mean completion length grows, the policy emits more reasoning-like text.
- **Will not see**: 'aha moments' of the R1 sort. The base is too small and 500 steps is too few. Full R1-Zero behaviors emerge over O(10K) steps on a 7B+ base.
- **Common failure**: format-reward hacking, where the policy emits `\\boxed{0}\\boxed{1}...\\boxed{42}` and gets credit for the last one. Mitigate with a stricter regex (`^.*\\boxed\{([^}]*)\}$`).
- **Stability**: if reward variance explodes, reduce learning rate to 5e-7. GRPO is sensitive on tiny models.

## Going further

- Move to Qwen2.5-1.5B or DeepSeek-R1-Distill-Qwen-1.5B as the base.
- Add a length-control term to the reward (penalize > N tokens) to head off the overthinking failure mode (Ch 6).
- Substitute a learned PRM for the verifier and inspect the reward-hacking surface (Ch 3).